# Text Search Engine Demo (BM25)

This notebook demonstrates Colette's lexical text search engine.

It uses project PDFs by default (`docs/pdf/*.pdf`).

Retrieval mode naming:
- `embedding_retrieval`: embedding retrieval
- `text_search_retrieval`: text search retrieval
- `hybrid`: combined retrieval


In [ ]:
import json
from pathlib import Path

import fitz

from colette.backends.tantivy import TantivyIndex, build_text_context
from colette.apidata import RAGObj, merge_rag_config


In [ ]:
class PrintLogger:
    def info(self, msg, *_):
        print(f"[INFO] {msg}")

    def debug(self, msg, *_):
        print(f"[DEBUG] {msg}")

    def warning(self, msg, *_):
        print(f"[WARN] {msg}")


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from start (default: cwd) until src/colette or pyproject.toml is found."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "colette").is_dir() or (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("Could not locate repo root (missing src/colette or pyproject.toml)")


def collect_example_pdfs(repo_root: Path):
    pdfs = []
    pdf_dir = repo_root / 'docs' / 'pdf'
    if pdf_dir.exists():
        pdfs.extend(sorted(pdf_dir.glob('*.pdf')))
    return pdfs


def build_documents_from_pdfs(pdf_files):
    """Build text-search documents from PDFs, mirroring rag_img.py exactly:
    - source  = str(full_path)   (same as str(document["source"]) in prod)
    - doc_id  = "<full_path>:page:<N>"
    - crop_label = None          (page-level docs, not crop-level)
    """
    docs = []
    for pdf_path in pdf_files:
        with fitz.open(pdf_path) as pdf_doc:
            for page_idx in range(pdf_doc.page_count):
                page_no = page_idx + 1
                text = pdf_doc.load_page(page_idx).get_text('text').strip()
                if not text:
                    continue
                docs.append({
                    'doc_id': f'{str(pdf_path)}:page:{page_no}',
                    'source': str(pdf_path),
                    'page_number': page_no,
                    'crop_label': None,
                    'label': None,
                    'content': text,
                })
    return docs


def load_rag_defaults(repo_root: Path):
    cfg_path = repo_root / 'src' / 'colette' / 'config' / 'vrag_default_index.json'
    with open(cfg_path, 'r', encoding='utf-8') as f:
        cfg = json.load(f)
    return RAGObj(**cfg['parameters']['input']['rag'])


In [ ]:
repo_root = find_repo_root()
pdf_files = collect_example_pdfs(repo_root)
if not pdf_files:
    raise RuntimeError('No PDFs found under docs/pdf')

rag_defaults = load_rag_defaults(repo_root)
print('Loaded RAG defaults from JSON config:')
print(f'  retrieval_mode            : {rag_defaults.retrieval_mode}')
print(f'  text_search_engine_top_k             : {rag_defaults.text_search_engine_top_k}')
print(f'  text_search_engine_fields            : {rag_defaults.text_search_engine_fields}')
print(f'  text_search_engine_max_chars_per_doc : {rag_defaults.text_search_engine_max_chars_per_doc}')
print(f'  text_search_engine_max_total_chars   : {rag_defaults.text_search_engine_max_total_chars}')
print()

# Index path mirrors the real pipeline: <app_repository>/mm_index/text_search_engine
app_dir = repo_root / 'app_colette'
index_path = app_dir / 'mm_index' / 'text_search_engine'

documents = build_documents_from_pdfs(pdf_files)
index = TantivyIndex(index_path, PrintLogger())
index.add_documents(documents, recreate=True)

print('PDF files:', [p.name for p in pdf_files])
print('Indexed docs:', len(documents))
print('Index path:', index_path)


In [ ]:
# Mirrors: self.text_search_engine_index.search(question, limit=rag_config.text_search_engine_top_k, fields=rag_config.text_search_engine_fields)
query = 'document title sources errors'
hits = index.search(query, limit=rag_defaults.text_search_engine_top_k, fields=rag_defaults.text_search_engine_fields)
print('Query:', query)
for i, h in enumerate(hits, 1):
    print(f"{i}. {Path(h['source']).name} page {h.get('page_number')} score={h['score']:.3f}")
    print(h['content'][:140].replace('\n', ' '), '...')
    print()


In [ ]:
# Mirrors: build_text_context(docs["text_context"], max_chars_per_doc=rag_config.text_search_engine_max_chars_per_doc,
#                                                   max_total_chars=rag_config.text_search_engine_max_total_chars)
bounded_hits, context_prompt = build_text_context(
    hits,
    max_chars_per_doc=rag_defaults.text_search_engine_max_chars_per_doc,
    max_total_chars=rag_defaults.text_search_engine_max_total_chars,
)

print('Hits before budget:', len(hits))
print('Hits after budget:', len(bounded_hits))
print('Prompt chars:', len(context_prompt))
print('\n----- Prompt block injected into LLM -----\n')
print(context_prompt)


In [ ]:
# Per-request override: merge_rag_config(service_rag, request_rag) in rag_img.py
# Service config comes from JSON defaults; request can override individual fields.
service_rag = rag_defaults                                        # loaded from vrag_default_index.json
request_rag = RAGObj(retrieval_mode='hybrid', text_search_engine_top_k=6)      # caller overrides mode + top_k
effective = merge_rag_config(service_rag, request_rag)

print('Service retrieval_mode :', service_rag.retrieval_mode)
print('Request retrieval_mode :', request_rag.retrieval_mode)
print('Effective retrieval_mode:', effective.retrieval_mode)
print('Effective text_search_engine_top_k :', effective.text_search_engine_top_k)
